# Email Appendix: EDA and Model Evidence

This appendix expands the email/message portion of the final prototype evidence notebook. The master reference remains `notebooks/00_final_prototype_evidence_notebook.ipynb`.

Purpose: document email dataset readiness, label balance, saved model metrics, artifact ownership, and dashboard traceability without rerunning slow training inside the notebook.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def project_path(relative: str) -> Path:
    return ROOT / relative

def load_json(relative: str) -> dict:
    path = project_path(relative)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def status(path: Path) -> str:
    return "PASS" if path.exists() else "MISSING"

ROOT

## Dataset Readiness

Email/message models are trained from processed text examples. The final notebook treats this as a text-classification channel: raw email/message sources are prepared into a consistent CSV, then TF-IDF features and candidate classifiers are evaluated.

In [ ]:
email_path = project_path("data/processed/email/email_dataset.csv")
email_df = pd.read_csv(email_path) if email_path.exists() else pd.DataFrame()

pd.DataFrame([
    {
        "Evidence item": "Processed email dataset",
        "Path": str(email_path.relative_to(ROOT)),
        "Status": status(email_path),
        "Rows": len(email_df),
        "Columns": len(email_df.columns),
        "Label column present": "label" in email_df.columns,
    }
])

In [ ]:
if not email_df.empty and "label" in email_df.columns:
    counts = email_df["label"].value_counts().sort_index()
    ax = counts.plot(kind="bar", figsize=(6, 4), color=["#2563EB", "#F97316"])
    ax.set_title("Email Label Distribution")
    ax.set_xlabel("Label")
    ax.set_ylabel("Rows")
    plt.tight_layout()
else:
    print("Processed email dataset or label column is unavailable.")

## Model Selection Evidence

The email workflow compares candidate TF-IDF classifiers such as Naive Bayes, Decision Tree, SVM, Random Forest, and XGBoost where available. The selected benchmark is based on saved validation metrics, not retraining during dashboard use.

In [ ]:
email_metrics = load_json("reports/metrics/email_model_metrics.json")
selected_name = email_metrics.get("best_model") or email_metrics.get("top_validation_model")
rows = []
for model_name, metrics in email_metrics.get("models", {}).items():
    rows.append({
        "Model": model_name,
        "Selected benchmark": "Yes" if model_name == selected_name else "No",
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1": metrics.get("f1"),
        "ROC-AUC": metrics.get("roc_auc"),
    })

metric_df = pd.DataFrame(rows)
metric_df.sort_values("F1", ascending=False) if not metric_df.empty else pd.DataFrame([{"Status": "No saved email metric rows found"}])

In [ ]:
if not metric_df.empty and "F1" in metric_df.columns:
    plot_df = metric_df.sort_values("F1", ascending=True)
    colors = ["#2563EB" if value == "Yes" else "#94A3B8" for value in plot_df["Selected benchmark"]]
    ax = plot_df.plot.barh(x="Model", y="F1", figsize=(7, 4), color=colors, legend=False)
    ax.set_title("Email Candidate Model F1 Scores")
    ax.set_xlabel("F1 score")
    plt.tight_layout()
else:
    print("No email model metrics available for plotting.")

## Runtime Artifact And Source Traceability

The dashboard loads saved artifacts. This appendix records the files that connect training output to the Email/message tab.

In [ ]:
artifact_paths = [
    "models/email_vectorizer.pkl",
    "models/email_nb.pkl",
    "models/email_dt.pkl",
    "models/email_svm.pkl",
    "models/email_rf.pkl",
    "models/email_xgb.pkl",
    "models/email_best.pkl",
    "reports/metrics/email_model_metrics.json",
    "app/email_tab.py",
    "src/training/email_trainer.py",
    "scripts/04_train_email_model.py",
]

pd.DataFrame([
    {"Path": relative, "Status": status(project_path(relative))}
    for relative in artifact_paths
])

## Reviewer Note

Email metrics help explain why the selected benchmark is useful, but they should not be presented as universal real-world accuracy. The Email/message tab remains an educational decision-support tool that highlights evidence for human review.